<a href="https://colab.research.google.com/github/zhe12345zhe/qwen2.5-1.5b-gptq-4bit-vllm-.ipynb/blob/main/qwen2_5_1_5b_gptq_4bit_vllm_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gptqmodel

In [ ]:
import gptqmodel

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.1.0
Transformers : 5.9.0
Torch        : 2.11.0+cu128
Triton       : 3.6.0


In [ ]:
import torch

In [ ]:
from gptqmodel import GPTQModel, QuantizeConfig

In [ ]:
from transformers import AutoTokenizer

In [ ]:
from datasets import load_dataset

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
quant_path = "./qwen2.5-1.5b-gptq-4bit"

In [ ]:
quant_config = QuantizeConfig(
    bits=4,
    group_size=128,          # 128
    damp_percent=0.01,
    desc_act=False,
    sym=True
)

INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_ly7vjih_`


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

这里的warning可以看到我的HF-token其实并没有完全配置好，但是因为qwen是开源的，没有配置好也可以获取其代码。

In [ ]:
model = GPTQModel.load(model_id, quant_config, trust_remote_code=True)

HF: overriding trust_remote_code=True to False for `Qwen/Qwen2.5-1.5B-Instruct` because model_type `qwen2` is integrated in installed transformers as `Qwen2ForCausalLM`.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  Loader: using checkpoint-backed lazy turtle source for `/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306`


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



INFO  Kernel: loaded -> `[]`                                                   


In [ ]:
dataset = load_dataset(
    "allenai/c4",
    data_files="en/c4-train.00001-of-01024.json.gz",
    split="train",
    # trust_remote_code 已不支持，直接去掉；如果报错，可改用本地简单文本
)

README.md:   0%|          | 0.00/41.1k [00:00<?, ?B/s]

en/c4-train.00001-of-01024.json.gz:   0%|          | 0.00/318M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
calibration_texts = dataset.select(range(256))["text"]
print(f"校准数据量: {len(calibration_texts)} 条")

校准数据量: 256 条


In [ ]:
model.quantize(calibration_texts, batch_size=1)

INFO  Packing Kernel: selected: `TorchLinear`                                  


INFO  Packing Kernel: selected: `TorchLinear`                                  


INFO  Calibration: Sort in descending order by length                          


INFO  Calibration: Total padded tokens: 0                                      


INFO  Calibration: Total non-padded tokens: 113914                             


...

INFO  | Finalize offload   | 196   | 0.008  | 0.017 | 3.291   | 0.2%   | model.layers.27.mlp.down_proj                     |


INFO  +--------------------+-------+--------+-------+---------+--------+---------------------------------------------------+


INFO  | Process finalize   | 1     | 0.001  | 0.001 | 0.001   | 0.0%   | gptq                                              |


INFO  +--------------------+-------+--------+-------+---------+--------+---------------------------------------------------+


{'gptq': [{'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.k_proj',
   'feat: in, out': '1536, 256',
   'dtype: size': 'bf16: 0.8MB',
   'loss': '0.0000126389',
   'samples': '113914',
   'damp': '0.01000',
   'time': '3.021',
   'fwd_time': '2.037',
   '(v)ram': 'cuda 0.56G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.v_proj',
   'feat: in, out': '1536, 256',
   'dtype: size': 'bf16: 0.8MB',
   'loss': '0.0000014281',
   'samples': '113914',
   'damp': '0.01000',
   'time': '3.021',
   'fwd_time': '2.037',
   '(v)ram': 'cuda 0.56G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.q_proj',
   'feat: in, out': '1536, 1536',
   'dtype: size': 'bf16: 4.6MB',
   'loss': '0.0000848390',
   'samples': '113914',
   'damp': '0.01000',
   'time': '3.084',
   'fwd_time': '2.037',
   '(v)ram': 'cuda 0.56G'},
  {'process': 'gptq',
   'layer': 0,
   'module': 'self_attn.o_proj',
   'feat: in, out': '1536, 1536',
   'dtype: size': 'bf16: 4.6MB',
   'loss': 

量化过程中显存占用峰值为8.13G

In [ ]:
# 保存量化模型和分词器
quant_path = "./qwen2.5-1.5b-gptq-4bit"   # 你可以自定义路径
model.save(quant_path)                     # 保存模型权重和配置
tokenizer.save_pretrained(quant_path)      # 保存分词器

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "method": "gptq",
  "quant_method": "gptq",
  "format": "gptq",
  "checkpoint_format": "gptq",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:7.1.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": 0.01,
    "damp_auto_increment": 0.01,
    "static_groups": false,
    "true_sequential": true,
    "mse": 0.0,
    "gptaq": null,
    "foem": null,
    "act_group_aware": true,
    "fallback": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": null
    },
    "offload_to_disk": true,
    "offload_to_disk_path": "/tmp/gptqmodel_ly7vjih_",
    "pack_impl": "cpu",
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "dense_vram_strategy": "exclusive",
    "dense_vram_strategy_devices": null,
    "moe_vram_strategy": "exclusive",
    "moe_vram_strateg

Files in directory:
quantize_config.json
generation_config.json
quant_log.csv
config.json
Content of saved `generation_config.json`:
{
    "bos_token_id": 151643,
    "do_sample": true,
    "eos_token_id": [
        151645,
        151643
    ],
    "repetition_penalty": 1.1,
    "temperature": 0.7,
    "top_k": 20,
    "top_p": 0.8,
    "transformers_version": "5.9.0"
}
Content of saved `config.json`:
{
    "architectures": [
        "Qwen2ForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 151643,
    "dtype": "bfloat16",
    "eos_token_id": 151645,
    "hidden_act": "silu",
    "hidden_size": 1536,
    "initializer_range": 0.02,
    "intermediate_size": 8960,
    "layer_types": [
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
        "full_attention",
    

INFO  Module: Re-tied embedding weights on shell model after lazy sync         


INFO  Module: Total direct tensors materialized from lazy checkpoint source: 85


...

INFO  | Finalize offload   | 196   | 0.008  | 0.017  | 3.291   | 0.2%   | model.layers.27.mlp.down_proj                     |


INFO  +--------------------+-------+--------+--------+---------+--------+---------------------------------------------------+


INFO  | Process finalize   | 1     | 0.001  | 0.001  | 0.001   | 0.0%   | gptq                                              |


INFO  +--------------------+-------+--------+--------+---------+--------+---------------------------------------------------+


('./qwen2.5-1.5b-gptq-4bit/tokenizer_config.json',
 './qwen2.5-1.5b-gptq-4bit/chat_template.jinja',
 './qwen2.5-1.5b-gptq-4bit/tokenizer.json')

In [ ]:
from safetensors import safe_open
import torch

# 直接指定模型文件路径（如果只有一个 safetensors 文件）
safetensors_path = "./qwen2.5-1.5b-gptq-4bit/model.safetensors"

with safe_open(safetensors_path, framework="pt", device="cpu") as f:
    # 获取第一个键名
    first_key = list(f.keys())[0]
    tensor = f.get_tensor(first_key)
    print(f"权重名称: {first_key}")
    print(f"形状: {tensor.shape}")
    print(f"数据类型: {tensor.dtype}")
    print("前 10 个数值:", tensor.flatten()[:10])

权重名称: model.embed_tokens.weight
形状: torch.Size([151936, 1536])
数据类型: torch.bfloat16
前 10 个数值: tensor([ 0.0063,  0.0123, -0.0099,  0.0135, -0.0216,  0.0081, -0.0027, -0.0151,
         0.0149, -0.0178], dtype=torch.bfloat16)


embed_tokens.weight是嵌入层，通常不会被量化

In [ ]:
from safetensors import safe_open
import torch

safetensors_path = "./qwen2.5-1.5b-gptq-4bit/model.safetensors"

with safe_open(safetensors_path, framework="pt", device="cpu") as f:
    # 查找第一个包含 'q_proj.weight' 的键
    target_key = None
    for key in f.keys():
        if 'model.layers.0.mlp.down_proj.qweight' in key:
            target_key = key
            break

    if target_key is None:
        print("未找到 model.layers.0.mlp.down_proj.qweight，列出所有键：")
        print(list(f.keys()))
    else:
        tensor = f.get_tensor(target_key)
        print(f"权重名称: {target_key}")
        print(f"形状: {tensor.shape}")
        print(f"数据类型: {tensor.dtype}")
        print("前 10 个数值:", tensor.flatten()[:10])
        if tensor.dtype == torch.int32:
            print("✓ 该层为打包的 int32 权重，量化成功。")
        elif tensor.dtype in (torch.uint8, torch.int8):
            print("✓ 该层为 int8/uint8 量化权重，量化成功。")
        else:
            print("✗ 该层仍为浮点类型，量化未正确应用。")

权重名称: model.layers.0.mlp.down_proj.qweight
形状: torch.Size([1120, 1536])
数据类型: torch.int32
前 10 个数值: tensor([-1771203416,  1681290693, -1984653641, -1383634823, -1665762639,
        -1433565081,  1367910520,  -930583719,  1202689928,  -910714232],
       dtype=torch.int32)
✓ 该层为打包的 int32 权重，量化成功。


layer0的Q权重成功进行了量化。看起来数据类型是int32，但是这其实4-bit权重打包后的存储格式，并非量化位宽是32-bit。4-bit 量化后，每个权重只占 4 位（0.5 字节）。为方便存储和计算，在 GPTQ 等 4-bit 量化实现中通常将连续的 8 个 4-bit 权重 拼成一个 int32 整数（32 位）。这样一来，存储形状变为 [8960/8, 1536] = [1120, 1536]。

In [ ]:
from google.colab import drive
import shutil
import os

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 量化模型本地路径
local_model_dir = "./qwen2.5-1.5b-gptq-4bit"

# 3. 云盘目标路径（可自定义）
drive_model_dir = "/content/drive/MyDrive/quantized_models/qwen2.5-1.5b-gptq-4bit"

# 4. 如果目标已存在，先删除（可选）
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)

# 5. 复制整个目录
shutil.copytree(local_model_dir, drive_model_dir)
print(f"模型已保存至 {drive_model_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
模型已保存至 /content/drive/MyDrive/quantized_models/qwen2.5-1.5b-gptq-4bit


In [ ]:
!pip install optimum

接下来我们使用CPU来推理。使用CPU主要是因为之前直接使用GPU推理时会编译marlin，而之前编译marlin总是不通过。当然为了更好展现模型量化的结果，使用vllm会有极快的速度与吞吐量。可惜我装vllm总是因为存在库不兼容而失败。为了展示一个初步的输出结果，我使用了CPU来推理。后续，我反复改变vllm与transformers的版本，得到了vllm==0.18与最新版transformers的组合最为稳定，且成功编译了marlin。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./qwen2.5-1.5b-gptq-4bit"   # 量化模型路径

# 强制 CPU 加载
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cpu",          # 使用 CPU
    trust_remote_code=True
)

# 简单测试
inputs = tokenizer("人工智能", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0]))

INFO  Kernel: Auto-selection: adding candidate `TorchAtenLinear`               


INFO  Kernel: selected -> `TorchAtenLinear`.                                   


Loading weights:   0%|          | 0/926 [00:00<?, ?it/s]

INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_1j9rj0fs`


INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.


INFO  Format: Converting GPTQ v1 to v2                                         


INFO  Optimize: `TorchAtenLinear` compilation triggered.                       


INFO  gc.collect() reclaimed 120 objects in 0.593s                             


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


人工智能的英文缩写是____。
A. AI
B. ASI
C. IAI
D. AIA
答案:
A

在进行电容与电感串联电路的测试时，应选用下列哪个仪器？
A


这里可以看到因为我没有给出规范的prompt，模型并没有理解我们的意图，只是根据给出的文本进行续写，下面我使用千问官方的prompt模板进行测试。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./qwen2.5-1.5b-gptq-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cpu",          # 暂时用 CPU 验证效果
    trust_remote_code=True
)

# 使用 Qwen2.5 官方聊天模板
def chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    # 构建消息列表
    messages = [
        {"role": "user", "content": prompt}
    ]
    # 应用模板：自动添加 <|im_start|>user ... <|im_end|> 和 <|im_start|>assistant
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True   # 添加 assistant 的开始标记
    )
    inputs = tokenizer(text, return_tensors="pt")
    # 移动输入到模型设备
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# 测试正常的问题
prompt = "请解释一下什么是人工智能？"
response = chat(prompt)
print("用户问题:", prompt)
print("模型回答:", response)

INFO  Kernel: Auto-selection: adding candidate `TorchAtenLinear`               


INFO  Kernel: selected -> `TorchAtenLinear`.                                   


Loading weights:   0%|          | 0/926 [00:00<?, ?it/s]

INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_pxdvn32x`


INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.


INFO  gc.collect() reclaimed 90 objects in 0.410s                              


用户问题: 请解释一下什么是人工智能？
模型回答: 人工智能（Artificial Intelligence，简称AI）是研究、开发用于模拟、扩展和增强人的智能的机器系统。它包括让计算机能够通过学习来自我改进，并且能够像人一样思考和行动。

人工智能可以分为三个主要部分：

1. **感知**：使计算机能够获取环境中的信息，比如视觉、听觉、触觉等。
2. **认知**：让计算机具备处理和理解这些信息的能力，包括分析、推理、判断以及学习新知识等。
3. **决策**：根据对周围世界的理解，做出符合预期的决定。

人工智能的应用广泛，涵盖了自动驾驶汽车、智能家居设备、医疗诊断辅助工具、金融服务中的自动化交易、教育领域的个性化辅导、艺术创作的生成引擎等众多领域。随着技术的发展，未来的人工智能可能会进一步深入我们的日常生活和工作，为人类带来更多的便利和支持。但同时也需要考虑其可能带来的隐私、安全和伦理问题。


接下来我尝试在vllm上部署量化后的模型。vllm与量化后的模型适配更好，可以实现更快的推理速度。

In [ ]:
import os
# 关键：提前禁用可能出问题的 kernel
os.environ["VLLM_USE_MARLIN"] = "0"  # 之前尝试部署量化后的模型时，问题都出现在编译marlin上，我们尝试禁用他再部署量化后的模型
os.environ["VLLM_USE_EXLLAMA"] = "0"
os.environ["VLLM_USE_TRITON"] = "0"
os.environ["VLLM_USE_ROCM"] = "0"

In [ ]:
#!pip uninstall vllm -y
#调试用，用于搭配下一行改变vllm版本。一开始我使用0.17.0，但是与transformers有冲突，我降级transformers后，transformers又与vllm有冲突，故升级vllm至0.18再尝试

Found existing installation: vllm 0.17.1
Uninstalling vllm-0.17.1:
  Successfully uninstalled vllm-0.17.1


In [ ]:
# 安装 vLLM（指定一个已知兼容的版本）
!pip install vllm==0.18.0
# 这里安装完报了一些error

  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 62.9 MB/s eta 0:00:00
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 13.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.37.2
    Uninstalling transformers-4.37.2:
      Successfully uninstalled transformers-4.37.2
  Attempting uninstall: xgrammar
    Found existing installation: xgrammar 0.1.29
    Uninstalling xgrammar-0.1.29:
      Successfully uninstalled xgrammar-0.1.29
  Attempting uninstall: flashinfer-python
   

这里我一开始是install了较为稳定的0.17版本vllm。但调用该版本vllm会出现error，具体是AttributeError: 'list' object has no attribute 'keys'。经过搜索，我了解这个问题是vllm的0.17.1版本与transformers最新版本不兼容导致的，所以我降级了transformers。但因为旧版transformers没有Gemma3Config，import vllm时会报错ImportError: cannot import name 'Gemma3Config' from 'transformers'，所以我又升级回了新版（因为新版Hugging Face 调整了模型架构的集成方式，不再需要 Gemma3Config 这个类），并尝试改变vllm的版本。最终发现最新版transformers与0.18版本的vllm搭配，最为稳定。

In [ ]:
!pip install --upgrade transformers

  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 50.2 MB/s eta 0:00:00
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.43.4
    Uninstalling transformers-4.43.4:
      Successfully uninstalled transformers-4.43.4
ERROR: pip's dependency resolver does not currently take into account all the packages that ar

In [ ]:
#用于改变transformers版本
#!pip uninstall transformers -y
#!pip install transformers==4.43.4

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gptqmodel 7.1.0 requires protobuf>=7.34.0, but you have protobuf 6.33.6 which is incompatible.
gptqmodel 7.1.0 requires transformers>=5.4.0, but you have transformers 4.43.4 which is incompatible.
vllm 0.18.0 requires tokenizers>=0.21.1, but you have tokenizers 0.19.1 which is incompatible

In [ ]:
# 在代码中导入
from vllm import LLM, SamplingParams

In [ ]:
model_path = "./qwen2.5-1.5b-gptq-4bit"

# Qwen 格式的 Prompt
prompt = "<|im_start|>user\n请简单解释一下什么是人工智能。<|im_end|>\n<|im_start|>assistant\n"

llm = LLM(model=model_path, trust_remote_code=True)
sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=256)
outputs = llm.generate([prompt], sampling_params)

print(outputs[0].outputs[0].text)

INFO 06-09 11:57:15 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': './qwen2.5-1.5b-gptq-4bit'}
INFO 06-09 11:57:48 [model.py:533] Resolved architecture: Qwen2ForCausalLM
WARNING 06-09 11:57:48 [model.py:1867] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-09 11:57:48 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 06-09 11:57:48 [model.py:1582] Using max model len 32768
INFO 06-09 11:57:48 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 06-09 11:57:48 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-09 11:57:48 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 06-09 11:57:50 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

人工智能（Artificial Intelligence，简称AI）是指由计算机系统所表现出的智能，包括学习、推理、感知、理解、判断和决策等智能活动。人工智能系统通过模拟、延伸和扩展人类智能，来完成诸如语言理解、图像识别、自然语言处理、语音识别、决策制定、自动化任务、异常检测、预测分析等任务。

人工智能的研究和应用涉及到计算机科学、工程学、心理学、生物学、神经科学等多个学科。在不同的领域和应用中，人工智能系统可以有多种不同的表现形式和应用目的，例如在医疗诊断、金融服务、交通运输、制造业自动化、教育、娱乐、游戏开发、安全监控、环境保护等领域都有广泛应用。

人工智能的发展和应用对推动科技进步、改善人们的生活质量、解决社会问题等都有重要意义。然而，人工智能系统也面临着一些挑战和风险，包括数据隐私和安全、伦理和道德问题、自动化和替代就业等。因此，需要在人工智能的发展和应用中加强规范和监管，以确保其健康发展和安全可靠。


我们的output达到了 86.75 toks/s，我认为这个速度对于T4这样的入门级GPU和1.5B参数量级而言已经非常理想。